# mtDNA (Abf2-Spot) pulldown — publication figure

Pipeline from MaxQuant `proteinGroups.txt` to the Abf2-Spot crosslink-dependence
figure: MaxQuant QC and unique-peptide filtering → per-sample iBAQ → log2 →
down-shifted-normal imputation of missing values → per-condition log2 ratio to
the untagged control (bait − control) → mitochondrial filter → figure.

The Abf2-Spot pulldown was an **exploratory, single-replicate** experiment
(native vs. +formaldehyde crosslink). Enrichment is reported as log2 fold change
over the matched untagged control **without significance testing**; key hits were
validated by immunoblot.

Run the one-time export cell (needs the annotation pickle) once to create the
lightweight `mito_genes.csv`; every later run reads that file.

## Design

| Condition | Bait sample | Control sample | Description |
|---|---|---|---|
| `Spot` | `iBAQ 342_S` | `iBAQ 380_S` | Abf2-Spot, native |
| `FA_Spot` | `iBAQ 342_SF` | `iBAQ 380_SF` | Abf2-Spot, + formaldehyde |

- **Direction:** positive log2FC = enriched in the Abf2-Spot pull-down relative to the untagged control.
- **Quantification:** iBAQ
- **Peptide filter:** ≥ 1 unique peptide. Note that with a single replicate, single-peptide identifications are uncorroborated.
- **Imputation:** down-shifted normal (1.8 SD below the column mean, width 0.3 SD; Perseus defaults), seeded (`SEED = 42`) so fold changes are reproducible. Estimated on the full filtered proteome, before the mitochondrial subset is taken.
- **Normalisation:** none applied between bait and control samples; fold changes are raw log2 differences. 
- **Scope:** the figure is restricted to the mitochondrial proteome (`DROP_NON_MITO = True`).

**Caveat on imputation with n = 1.** Where a protein is absent from one sample,
its imputed value is the entire measurement for that condition, so the resulting
fold change is a lower bound set by the imputation parameters rather than a
measured ratio. This is standard practice, but it is why the analysis is framed
as exploratory and why key hits were confirmed by immunoblot.

## Running this notebook

Set `DATA_DIR` in the config cell to the directory containing
`proteinGroups.txt` and `mito_genes.csv`.

**Requires:** `pandas`, `numpy`, `matplotlib`; `adjustText` optional.

**Note:** `mito_genes.csv` must exist before the config cell runs. Commit it
alongside the notebook, or run the export cell first.


## 1. Setup

Library imports and global figure settings.

| Library | Role in this notebook |
|---|---|
| `pandas` | Reading the MaxQuant `proteinGroups` table and reshaping iBAQ data |
| `numpy` | Log transformation, imputation, and array operations |
| `matplotlib` | Figure generation |
| `matplotlib.lines.Line2D` | Custom legend handles for protein-class colouring |
| `adjustText` | Optional; repositions overlapping gene labels. Falls back with a warning if unavailable |
| `ast` | Parsing MultiIndex column headers from the annotation table |
| `os` | File existence checks for the cached mitochondrial gene list |

**Figure settings:** `pdf.fonttype: 42` keeps text editable in the exported PDF
rather than converting it to outlines, so labels can be adjusted during figure
assembly. Top and right spines removed; base font size 9 pt.

In [ ]:
import os, ast, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
try:
    from adjustText import adjust_text
except ImportError:
    adjust_text=None; warnings.warn("adjustText not installed -> labels may overlap (pip install adjustText)")
plt.rcParams.update({"font.size":9,"axes.spines.top":False,"axes.spines.right":False,"pdf.fonttype":42})

## 2. Configuration and protein classes

**Input.** Raw MaxQuant `proteinGroups.txt`, plus a cached mitochondrial gene
list (`mito_genes.csv`) derived from the Morgenstern high-confidence proteome.
The cache is generated once by the export cell below and read directly on every
subsequent run, so the notebook does not require the full annotation pickle.

**Contrasts.** Two bait/control pairs, each a single sample:

| Condition | Bait | Control | Description |
|---|---|---|---|
| `Spot` | `iBAQ 342_S` | `iBAQ 380_S` | Abf2-Spot, native |
| `FA_Spot` | `iBAQ 342_SF` | `iBAQ 380_SF` | Abf2-Spot, + formaldehyde crosslink |

**Parameters:**

| Parameter | Value | Meaning |
|---|---|---|
| `MIN_UNIQUE_PEPTIDES` | 1 | Minimum unique peptides for a protein group to be retained |
| `IMP_DOWNSHIFT` / `IMP_WIDTH` | 1.8 / 0.3 | Down-shifted normal imputation (Perseus defaults), in column SD units |
| `SEED` | 42 | Fixes the imputation draw so fold changes are reproducible |
| `DROP_NON_MITO` | `True` | Restrict the figure to the mitochondrial proteome |

**Protein classes.** Genes are assigned to functional classes, tested in
dictionary order — **first match wins**, so genes in more than one list take the
class listed earlier. Classes are then collapsed into three narrative display
groups: PDH complex, Nucleoid / mtDNA maintenance, and Mitochondrial gene
expression (mitoribosome, translation factors, mtRNA metabolism, and
mtDNA-encoded products combined). Mitochondrial proteins without a class fall
into "other mitochondrial" and form the grey-blue background.

Marker size increases with narrative prominence, with the PDH complex drawn
largest.

In [ ]:
# ============================== CONFIG ==============================
# Directory containing the MaxQuant output and the cached mito-gene list.
DATA_DIR  = '.'
INFILE    = f"{DATA_DIR}/proteinGroups.txt"      # raw MaxQuant output
MITO_FILE = f"{DATA_DIR}/mito_genes.csv"         # lightweight mito-gene list (created by the export cell below)

# Abf2-Spot pulldown -> (bait iBAQ column, control iBAQ column)
CONDITIONS = {
    "Spot":    ("iBAQ 342_S",  "iBAQ 380_S"),    # Abf2-Spot, native
    "FA_Spot": ("iBAQ 342_SF", "iBAQ 380_SF"),   # Abf2-Spot, + formaldehyde
}
GENE_COL = "Gene names"; ID_COL = "Protein IDs"

MIN_UNIQUE_PEPTIDES = 1     # set to 2 for a stricter list
IMP_DOWNSHIFT = 1.8         # down-shifted-normal imputation (Perseus defaults)
IMP_WIDTH     = 0.3
SEED          = 42
DROP_NON_MITO = True
# ===================================================================

In [ ]:
# ---- protein classes + colours; load MITO_GENES from the lightweight csv ----
CLASSES = {
    "PDH complex":            ["PDA1","PDB1","LAT1","LPD1","PDX1","PKP1","PKP2"],
    "mtDNA / nucleoid":       ["ABF2","ACO1","ATP1","CHA1","ECM10","HSP60","IDH1","IDP1","ILV5","ILV6",
                               "KGD1","KGD2","LPD1","LSC1","MGM101","MNP1","PDA1","PDB1","RIM1","RPO41","SLS1",
                               "SSC1","YHM2","MIP1","MSH1","CCE1","IRC3"],
    "Large Ribosomal Subunit": ["IMG1","IMG2","MHR1","MNP1","MRP20","MRP35","MRP49","MRP7",
                                "MRPL1","MRPL10","MRPL11","MRPL13","MRPL15","MRPL16","MRPL17","MRPL19",
                                "MRPL20","MRPL22","MRPL23","MRPL24","MRPL25","MRPL27","MRPL28","MRPL3",
                                "MRPL31","MRPL32","MRPL33","MRPL35","MRPL36","MRPL37","MRPL38","MRPL39",
                                "MRPL4","MRPL40","MRPL44","MRPL49","MRPL50","MRPL51","MRPL6","MRPL7",
                                "MRPL8","MRPL9","MRX14","PTH4","RML2","RTC6","YML6"],
    "Small Ribosomal Subunit": ["EHD3","FYV4","MRP1","MRP10","MRP13","MRP17","MRP2","MRP21","MRP4","MRP51",
                                "MRPS12","MRPS16","MRPS17","MRPS18","MRPS28","MRPS35","MRPS5","MRPS8","MRPS9",
                                "NAM9","PET123","PPE1","QRI5","RSM10","RSM18","RSM19","RSM22","RSM23","RSM24",
                                "RSM25","RSM26","RSM27","RSM28","RSM7","SWS2","VAR1"],
    "TCA":["ACO1","ACO2","CIT1","CIT2","CIT3","FUM1","IDH1","IDH2","IDP1","IDP2","IDP3",
           "KGD1","KGD2","KGD4","LPD1","LSC1","LSC2","MDH1","MDH2","MDH3","SDH1","SDH2",
           "SDH3","SDH4","SDH5","SDH9","SHH3","SHH4"],
    "Trans. Factors":['RSO55','SLS1','HER2','DPC29','SOV1','MEF2','NAM2','MEF1','RRF1','IFM1','PET54','CCM1','MTG2','RMD9',
                      'MSR1','ISM1','MMF1','GTF1','MTF2','HTS1','PET112','MSK1','MRF1','MSE1','TUF1','PTH1','PET127','AEP3','NAM1'],
    "mtRNA metabolism":['MRS1','RPM2','SUV3','DSS1','CBP1','CBP2','CBP3','CBP6','MSS18','MNE1','MRM1','MRM2',
                        'RRG9','CBT1','PET309','CBS1','CBS2','ATP22','PET111','PET122','PET494','MSS51','AEP1','AEP2',
                        'MST1','MSD1','MSY1','MSW1','MSM1','MSF1','SLM5','GRS1','VAS1',
                        'PUS6','MTO1','MSS1','TRM5','TRZ1'],
    "mtDNA encoded":['COX1','COX2','COX3','ATP6','ATP8','ATP9','COB','VAR1'],
}
def classify(g):
    g=str(g).upper()
    for cls,genes in CLASSES.items():
        if g in genes: return cls
    return "other"

STORY_MAP = {
    "PDH complex":              "PDH complex",
    "mtDNA / nucleoid":         "Nucleoid / mtDNA maintenance",
    "Large Ribosomal Subunit":  "Mitochondrial gene expression",
    "Small Ribosomal Subunit":  "Mitochondrial gene expression",
    "Trans. Factors":           "Mitochondrial gene expression",
    "mtRNA metabolism":         "Mitochondrial gene expression",
    "mtDNA encoded":            "Mitochondrial gene expression",
}
DISPLAY_ORDER  = ["other mitochondrial", "Mitochondrial gene expression",
                  "Nucleoid / mtDNA maintenance", "PDH complex"]
DISPLAY_COLORS = {"other mitochondrial":"#9aabb8", "Mitochondrial gene expression":"#e9773b",
                  "Nucleoid / mtDNA maintenance":"#2a9d8f", "PDH complex":"#d1495b",
                  "non-mitochondrial":"#dddddd"}

# marker size per group (PDH deliberately larger so it stands out)
SIZE = {"PDH complex":78, "Nucleoid / mtDNA maintenance":40,
        "Mitochondrial gene expression":40, "other mitochondrial":14}

MITO_GENES = set(pd.read_csv(MITO_FILE)["gene"].astype(str).str.upper())
print(f"MITO_GENES: {len(MITO_GENES)} (from {MITO_FILE})")

## 3. Load, filter, transform, and annotate

**1. MaxQuant filtering.** Reverse hits, potential contaminants, and protein
groups identified only by a modification site are removed, followed by a
minimum unique-peptide filter. Display labels are taken as the first entry of
each group's semicolon-separated gene name list, falling back to the first
protein ID where no gene name is annotated.

**2. Log2 transformation and imputation.** iBAQ intensities are parsed to
numeric; zeros encode non-detection and become `NaN` before transformation, so
they are treated as missing rather than as measured zeros. Missing values are
then imputed per column from a normal distribution down-shifted 1.8 SD below
the observed column mean with width 0.3 SD, using observed values only to
estimate the mean and SD. Imputation is performed on the full filtered
proteome, before the mitochondrial subset is taken, so the imputation
distribution reflects the whole intensity range of each sample. The draw is
seeded, so fold changes are identical across runs.

**3. Background correction.** For each condition, the reported value is
log2(bait) − log2(control) for the matched sample pair, i.e. the log2 ratio of
bait to control iBAQ.

**4. Annotation and filtering.** Each protein is assigned a functional class and
display group, and flagged against the mitochondrial reference set. With
`DROP_NON_MITO = True`, non-mitochondrial proteins are removed at this point,
so the figure shows enrichment within the mitochondrial proteome only.

In [ ]:
# ---- 1) load + standard MaxQuant filtering ----
df = pd.read_csv(INFILE, sep="\t", low_memory=False)
n_raw = len(df)
for c in ["Reverse","Potential contaminant","Only identified by site"]:
    if c in df.columns: df = df[df[c] != "+"]
if "Unique peptides" in df.columns:
    df = df[pd.to_numeric(df["Unique peptides"],errors="coerce").fillna(0) >= MIN_UNIQUE_PEPTIDES]
df = df.reset_index(drop=True)
df[GENE_COL] = df[GENE_COL].fillna("").astype(str)
df["label"]  = df[GENE_COL].str.split(";").str[0]
df.loc[df["label"]=="", "label"] = df[ID_COL].str.split(";").str[0]
print(f"{n_raw} -> {len(df)} protein groups after filtering")

In [ ]:
# ---- 2) per-sample log2(iBAQ) + down-shifted-normal imputation ----
samples = sorted({c for pair in CONDITIONS.values() for c in pair})
missing = [c for c in samples if c not in df.columns]
assert not missing, f"iBAQ columns not found: {missing}"
L = df[samples].apply(lambda s: pd.to_numeric(s, errors="coerce"))
L = np.log2(L.replace(0, np.nan))
rng = np.random.default_rng(SEED)
for c in L.columns:
    col = L[c].astype(float); m, s = col.mean(), col.std()
    miss = col.isna().values
    if miss.any():
        L.loc[miss, c] = rng.normal(m - IMP_DOWNSHIFT*s, IMP_WIDTH*s, int(miss.sum()))

# ---- 3) background-corrected log2 fold change per condition ----
for name,(bait,ctrl) in CONDITIONS.items():
    df[name+" - contr"] = L[bait].values - L[ctrl].values
diff_cols = [f"{n} - contr" for n in CONDITIONS]

# ---- 4) annotate + mito filter ----
df["gene_u"] = df["label"].str.upper()
df["class"]  = df["gene_u"].map(classify)
df["mito"]   = df["gene_u"].isin(MITO_GENES)
_story = df["class"].map(STORY_MAP)
df["display_group"] = np.where(_story.notna(), _story,
                        np.where(df["mito"].values, "other mitochondrial", "non-mitochondrial"))
if DROP_NON_MITO:
    df = df[df["mito"]].copy()
print(df["display_group"].value_counts().to_string())

## 4. Crosslink-dependence figure

Compares enrichment in the native pull-down (x-axis) against the
formaldehyde-crosslinked pull-down (y-axis), both as background-corrected log2
fold change over their matched controls. Proteins retained only under
crosslinking sit above the diagonal; proteins recovered equally in both lie
along it.

**Plot elements:**

- **Diagonal reference line** (y = x): equal enrichment in both conditions.
  Vertical distance from this line is the crosslink-dependence of the
  interaction, which is the quantity the figure is built to show.
- **Zero lines** mark parity with the control for each axis.
- **Equal aspect ratio and shared axis limits**, so displacement from the
  diagonal is read at true scale in both directions.
- **Draw order and styling** run background-to-foreground: other mitochondrial
  proteins are small, pale and semi-transparent; narrative groups are larger and
  opaque; PDH complex subunits are largest with black edges.
- **Labels** are applied to all PDH complex subunits and the five
  nucleoid/mtDNA-maintenance proteins with the highest crosslinked fold change.
  `adjustText` de-overlaps them where available.

Axis limits are taken from the data range with 0.5 log2 units of padding, so no
points are clipped.

In [ ]:
# ---- crosslink-dependence figure: native (x) vs +formaldehyde (y) ----
def crosslink_figure(native, fa, title, fname):
    fig, ax = plt.subplots(figsize=(5.6, 5.4))
    for grp in DISPLAY_ORDER:
        s = df[df["display_group"]==grp]
        if s.empty: continue
        pdh = grp=="PDH complex"
        ax.scatter(s[native+" - contr"], s[fa+" - contr"], s=SIZE[grp], c=DISPLAY_COLORS[grp],
                   alpha=(0.35 if grp=="other mitochondrial" else 0.95),
                   edgecolor=("black" if pdh else "white" if grp!="other mitochondrial" else "none"),
                   linewidth=(0.8 if pdh else 0.3), zorder=(6 if pdh else 4 if grp!="other mitochondrial" else 1),
                   label=grp)
    lo = min(df[native+" - contr"].min(), df[fa+" - contr"].min()) - 0.5
    hi = max(df[native+" - contr"].max(), df[fa+" - contr"].max()) + 0.5
    ax.plot([lo,hi],[lo,hi],"--",color="0.55",lw=0.8,zorder=0)
    ax.axhline(0,color="0.85",lw=0.6,zorder=0); ax.axvline(0,color="0.85",lw=0.6,zorder=0)
    ax.set_xlim(lo,hi); ax.set_ylim(lo,hi); ax.set_aspect("equal")
    ax.set_xlabel(f"log$_2$FC  {native} (native)")
    ax.set_ylabel(f"log$_2$FC  {fa} (+crosslink)")
    ax.set_title(title, fontweight="bold")
    # label PDH subunits + the top 5 nucleoid/maintenance hits (by crosslinked FC)
    pdh_lab  = df[df["display_group"] == "PDH complex"]
    nuc_lab  = (df[df["display_group"] == "Nucleoid / mtDNA maintenance"]
                  .nlargest(5, fa+" - contr"))
    lab = pd.concat([pdh_lab, nuc_lab]).drop_duplicates("label")
    texts = [ax.text(r[native+" - contr"], r[fa+" - contr"], r["label"], fontsize=7,
                     fontstyle="italic", color=DISPLAY_COLORS[r["display_group"]])
             for _, r in lab.iterrows()]
    if adjust_text and texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.4))
    ax.legend(frameon=False, fontsize=7, loc="upper left")
    fig.tight_layout()
    #fig.savefig(fname+".pdf", bbox_inches="tight"); fig.savefig(fname+".png", dpi=300, bbox_inches="tight")
    plt.show(); print("saved", fname+".pdf / .png")

# Figure — Abf2-Spot
crosslink_figure("Spot", "FA_Spot", "Abf2-Spot", "abf2_spot_crosslink")

## 5. Export enrichment table

Writes the mitochondrial-filtered enrichment table, sorted by crosslinked fold
change. Each row gives the gene label, display group, and background-corrected
log2 fold change in the native and crosslinked conditions.

Fold changes are ratios from single samples per condition; see the header for
the caveats on imputation and the absence of significance testing.

In [ ]:
# ---- export the mito-filtered Abf2-Spot enrichment table (supplementary) ----
out = (df[["label", "display_group", "Spot - contr", "FA_Spot - contr"]]
       .rename(columns={"label": "gene",
                        "Spot - contr":    "log2FC_native",
                        "FA_Spot - contr": "log2FC_crosslink"})
       .sort_values("log2FC_crosslink", ascending=False))
out.to_csv(f"{DATA_DIR}/abf2_spot_enrichment.csv", index=False)
print(f"wrote abf2_spot_enrichment.csv  ({len(out)} mitochondrial proteins)")